# sfmc_sync_re_email_status
Syncs SFMC opt-out / bounce flags → Blackbaud SKY API communication preferences + actions.

Source datasets:
- `raw_re_emailaddresses`  — RE NXT email address records
- `bronze_re_constituents` — transformed constituents layer
- `085c865f-2a21-4f04-9333-8e5c24789a6b` — SFMC bounce events (UUID; replace name if available)
- `OUT | All SFMC opt outs (wide)` — SFMC optouts table

In [ ]:
%run user_configuration.ipynb
%run renxt_core.ipynb
import pandas as pd


In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
RE_EMAIL_ADDRESSES_DATASET = "raw_re_email_addresses"
                              # Domo dataset name — note underscore between email and addresses
                              # NOTE: no underscore between email and addresses —
                              # matches the canonical endpoint name "emailaddresses"
RE_CONSTITUENTS_DATASET    = "bronze_re_constituents"
SFMC_BOUNCE_EVENTS_DATASET = "085c865f-2a21-4f04-9333-8e5c24789a6b"
                              # SFMC bounce events — UUID (replace with name if renamed in Domo)
SFMC_OPTOUTS_DATASET       = "OUT | All SFMC opt outs (wide)"

COMM_PREF_URL    = f"{API_BASE}/constituent/v1/communicationpreferences"
ACTIONS_URL      = f"{API_BASE}/constituent/v1/actions"
ACTION_CF_URL    = f"{API_BASE}/constituent/v1/actions/customfields"
AUTHOR_EMAIL     = "data_insights@heartfoundation.org.au"  # update as needed
DRY_RUN          = True   # set False to send actual API calls
# ─────────────────────────────────────────────────────────────────────────────


In [ ]:
# Load all source datasets
print("Loading datasets...")
raw_email_df    = domo.read_dataframe(RE_EMAIL_ADDRESSES_DATASET,  query="SELECT * FROM table")
constituents_df = domo.read_dataframe(RE_CONSTITUENTS_DATASET,     query="SELECT * FROM table")
bounce_events_df= domo.read_dataframe(SFMC_BOUNCE_EVENTS_DATASET,  query="SELECT * FROM table")
optouts_df      = domo.read_dataframe(SFMC_OPTOUTS_DATASET,        query="SELECT * FROM table")
print(f"  raw_email_df    : {len(raw_email_df):,}")
print(f"  constituents_df : {len(constituents_df):,}")
print(f"  bounce_events_df: {len(bounce_events_df):,}")
print(f"  optouts_df      : {len(optouts_df):,}")


In [ ]:
# ── Helpers ────────────────────────────────────────────────────────────────────

def normalize_email(s):
    if isinstance(s, pd.Series):
        return s.astype("string").str.strip().str.lower().fillna("")
    return str(s).strip().lower() if s and not pd.isna(s) else ""

def to_bool(v) -> bool:
    if isinstance(v, bool): return v
    if isinstance(v, str):  return v.strip().upper() in ("TRUE","Y","YES","1")
    return bool(v) if v is not None else False

def get_unsubscribe_emails(df: pd.DataFrame) -> set:
    cols      = [c for c in ["Revenue_optout","Manual_optout","Global_optout"] if c in df.columns]
    email_col = next((c for c in ["Email Address","Email","email"] if c in df.columns), None)
    if not cols or not email_col: return set()
    mask = df[cols].apply(lambda c: c.astype(str).str.upper() == "Y").any(axis=1)
    return set(normalize_email(df.loc[mask, email_col]))

def get_inactive_emails(df: pd.DataFrame) -> set:
    col       = next((c for c in ["Held_and_hard_bounce_optout"] if c in df.columns), None)
    email_col = next((c for c in ["Email Address","Email","email"] if c in df.columns), None)
    if not col or not email_col: return set()
    mask = df[col].astype(str).str.upper() == "Y"
    return set(normalize_email(df.loc[mask, email_col]))

def bounce_email_set(df: pd.DataFrame, event_type: str) -> set:
    if "EventType" not in df.columns: return set()
    key_col = next((c for c in ["SubscriberKey","EmailAddress","email"] if c in df.columns), None)
    if not key_col: return set()
    return set(normalize_email(df.loc[df["EventType"] == event_type, key_col]))


In [ ]:
# Build target flag sets
unsubscribe_emails = get_unsubscribe_emails(optouts_df)
inactive_emails    = get_inactive_emails(optouts_df) | bounce_email_set(bounce_events_df, "HardBounce")

print(f"Emails to unsubscribe  : {len(unsubscribe_emails):,}")
print(f"Emails to mark inactive: {len(inactive_emails):,}")


In [ ]:
# Build work queue: constituents whose email flags need changing
avail      = [c for c in ["email_address","email_do_not_email","email_inactive","id"]
              if c in constituents_df.columns]
status_df  = constituents_df[avail].copy()
status_df["email_address"]       = normalize_email(status_df.get("email_address", pd.Series(dtype=str)))
status_df  = status_df.dropna(subset=["email_address"]).query("email_address != ''")
status_df["constituent_id"]      = status_df["id"].astype(str).str.strip()
status_df["current_do_not_email"]= status_df.get("email_do_not_email", pd.Series(False)).apply(to_bool)
status_df["current_inactive"]    = status_df.get("email_inactive",    pd.Series(False)).apply(to_bool)
status_df["target_do_not_email"] = status_df["email_address"].isin(unsubscribe_emails)
status_df["target_inactive"]     = status_df["email_address"].isin(inactive_emails)
status_df["requires_update"]     = (
    (status_df["current_do_not_email"] != status_df["target_do_not_email"])
    | (status_df["current_inactive"]   != status_df["target_inactive"])
)
print(f"Constituents requiring update: {status_df['requires_update'].sum():,}")


In [ ]:
# Build unsubed queue — join to email address IDs for POST
if "address" in raw_email_df.columns:
    raw_email_df["email_address"] = normalize_email(raw_email_df["address"])
    unsubed_map = (
        raw_email_df[raw_email_df["email_address"].isin(unsubscribe_emails)]
        [["email_address","constituent_id"]]
        .dropna(subset=["constituent_id"])
        .assign(constituent_id=lambda d: d["constituent_id"].astype(str).str.strip())
        .drop_duplicates(subset=["constituent_id"])
        .reset_index(drop=True)
    )
else:
    unsubed_map = pd.DataFrame(columns=["email_address","constituent_id"])

print(f"Unique constituents to POST comm preference: {len(unsubed_map):,}")


In [ ]:
# POST communication preferences and action audit records
if DRY_RUN:
    print("⚠️  DRY_RUN=True — no API calls sent.")
    print("Sample queue (first 10 rows):")
    print(unsubed_map.head(10).to_string(index=False))
else:
    from datetime import datetime as _dt

    token_mgr = TokenManager(interactive=False)
    sess      = requests.Session()
    today     = _dt.now().strftime("%Y-%m-%dT00:00:00Z")

    comm_results   = []
    action_results = []

    for _, row in unsubed_map.iterrows():
        cid = row["constituent_id"]

        # POST communication preference
        cp_resp = api_request_with_auth(
            "POST", COMM_PREF_URL, token_mgr=token_mgr, session=sess,
            json_body={"constituent_id": cid, "solicit_code": "Do Not Email",
                       "start": None, "end": None},
        )
        comm_results.append({"constituent_id": cid, "status": cp_resp.status_code})

        if cp_resp.ok:
            # POST audit action
            act_resp = api_request_with_auth(
                "POST", ACTIONS_URL, token_mgr=token_mgr, session=sess,
                json_body={"constituent_id": cid, "author": AUTHOR_EMAIL,
                           "category": "Task/Other", "completed": True,
                           "completed_date": today, "date": today,
                           "description": "Do Not Email added as opted out in SFMC",
                           "direction": "Outbound", "outcome": "Successful",
                           "priority": "Normal", "status": "Completed",
                           "summary": "Solicit code added", "type": "Solicit Code Change"},
            )
            action_id = (act_resp.json().get("id","") if act_resp.ok else "")
            action_results.append({"constituent_id": cid,
                                   "action_status": act_resp.status_code,
                                   "action_id": action_id})

    comm_df   = pd.DataFrame(comm_results)
    action_df = pd.DataFrame(action_results)
    print(f"\nComm prefs posted : {(comm_df['status']   < 300).sum():,} / {len(comm_df):,}")
    print(f"Actions posted    : {(action_df['action_status'] < 300).sum():,} / {len(action_df):,}")
